# Entraînement de modèles — ElectricityLoadDiagrams20112014

Objectif : construire un pipeline pédagogique complet :

**dataset → nettoyage → split temporel → fit → predict → évaluation → comparaison de modèles / hyperparamètres → choix du meilleur modèle**

> Important : comme il s'agit d'une série temporelle, on ne mélange pas aléatoirement passé et futur.


## 1. Imports et chemins

On charge les données préparées dans `data/modelling/`.


In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path("data/modelling")
FEATURES_PATH = DATA_DIR / "features.parquet"
TARGET_PATH = DATA_DIR / "target.parquet"

TARGET = "consumption_kwh"
RANDOM_STATE = 42


## 2. Chargement du dataset

Les Parquet contiennent déjà :
- les **features** temporelles (`lag_1d`, `lag_7d`, etc.) ;
- la **cible** `consumption_kwh`.

On convertit les valeurs numériques en `float32` pour limiter la RAM.


In [ ]:
df_features = pq.read_table(FEATURES_PATH).to_pandas()
df_target = pq.read_table(TARGET_PATH).to_pandas()

df_features = df_features.astype("float32")
df_target = df_target.astype("float32")

print("Features :", df_features.shape)
print("Target   :", df_target.shape)
print("Index identique :", df_features.index.equals(df_target.index))
df_features.head()


## 3. Fusion et nettoyage

Les `NaN` au début de la série sont normaux : par exemple `lag_365d` ne peut pas exister avant d'avoir un an d'historique.

On les retire pour l'entraînement.


In [ ]:
df = df_features.join(df_target, how="inner")
del df_features, df_target
gc.collect()

print("Avant dropna :", df.shape)
df = df.dropna()
print("Après dropna :", df.shape)

FEATURES = [c for c in df.columns if c != TARGET]
print("Features utilisées :", FEATURES)


## 4. Observer les années réellement disponibles

Avec `lag_365d`, l'année 2011 peut disparaître presque entièrement après `dropna()`.

C'est important : un split `2011-2012 / 2013-2014` donnerait alors beaucoup plus de données au test qu'au train.


In [ ]:
years = df.index.get_level_values("timestamp").year
year_counts = pd.Series(years).value_counts().sort_index()
year_counts


## 5. Split temporel : train / validation / test

Pour une série temporelle :

- **2012 → train**
- **2013 → validation** (choix du modèle / hyperparamètres)
- **2014 → test final**

Ainsi, le modèle ne voit jamais le futur pendant l'entraînement.

On évite donc ici `train_test_split(..., shuffle=True)`.


In [ ]:
train_mask = years == 2012
valid_mask = years == 2013
test_mask = years == 2014

X_train = df.loc[train_mask, FEATURES].to_numpy(dtype=np.float32)
y_train = df.loc[train_mask, TARGET].to_numpy(dtype=np.float32)

X_valid = df.loc[valid_mask, FEATURES].to_numpy(dtype=np.float32)
y_valid = df.loc[valid_mask, TARGET].to_numpy(dtype=np.float32)

X_test = df.loc[test_mask, FEATURES].to_numpy(dtype=np.float32)
y_test = df.loc[test_mask, TARGET].to_numpy(dtype=np.float32)

del df, years, train_mask, valid_mask, test_mask
gc.collect()

print("Train :", X_train.shape, y_train.shape)
print("Valid :", X_valid.shape, y_valid.shape)
print("Test  :", X_test.shape, y_test.shape)


## 6. Limiter la charge du TP

Le dataset contient plusieurs millions de lignes. Une Random Forest de centaines d'arbres sur toutes les lignes peut consommer énormément de RAM.

Pour le TP, on garde un sous-échantillon reproductible **à l'intérieur de chaque période**.

Mettre `MAX_*_ROWS = None` pour utiliser toutes les lignes si la machine le permet.


In [ ]:
MAX_TRAIN_ROWS = 800_000
MAX_VALID_ROWS = 500_000
MAX_TEST_ROWS = 1_000_000

rng = np.random.default_rng(RANDOM_STATE)

def sample_xy(X, y, max_rows):
    if max_rows is None or len(X) <= max_rows:
        return X, y
    idx = rng.choice(len(X), size=max_rows, replace=False)
    return X[idx], y[idx]

X_train_fit, y_train_fit = sample_xy(X_train, y_train, MAX_TRAIN_ROWS)
X_valid_eval, y_valid_eval = sample_xy(X_valid, y_valid, MAX_VALID_ROWS)
X_test_eval, y_test_eval = sample_xy(X_test, y_test, MAX_TEST_ROWS)

print("Sous-échantillon train :", X_train_fit.shape)
print("Sous-échantillon valid :", X_valid_eval.shape)
print("Sous-échantillon test  :", X_test_eval.shape)


## 7. Fonction d'évaluation

- **MAE** : erreur absolue moyenne ;
- **RMSE** : pénalise davantage les grosses erreurs ;
- **R²** : part de variance expliquée (plus proche de 1 = mieux).


In [ ]:
def evaluate(name, model, X, y):
    pred = model.predict(X)
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    r2 = r2_score(y, pred)
    return {
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }


## 8. Plusieurs modèles + hyperparamètres

Les **paramètres** sont appris par `.fit()`.

Les **hyperparamètres** sont choisis avant le training :
- `alpha` pour Ridge ;
- `n_estimators`, `max_depth`, `min_samples_leaf`, `max_samples` pour Random Forest ;
- `learning_rate`, `max_iter`, `max_leaf_nodes` pour HistGradientBoosting.

La liste ci-dessous constitue une petite boucle d'expérimentation.


In [ ]:
candidates = [
    (
        "Dummy(mean)",
        DummyRegressor(strategy="mean"),
    ),
    (
        "Ridge alpha=1",
        Ridge(alpha=1.0),
    ),
    (
        "Ridge alpha=10",
        Ridge(alpha=10.0),
    ),
    (
        "RandomForest depth=10",
        RandomForestRegressor(
            n_estimators=40,
            max_depth=10,
            min_samples_leaf=20,
            max_samples=0.30,
            n_jobs=2,
            random_state=RANDOM_STATE,
        ),
    ),
    (
        "RandomForest depth=16",
        RandomForestRegressor(
            n_estimators=60,
            max_depth=16,
            min_samples_leaf=10,
            max_samples=0.30,
            n_jobs=2,
            random_state=RANDOM_STATE,
        ),
    ),
    (
        "HistGradientBoosting",
        HistGradientBoostingRegressor(
            learning_rate=0.08,
            max_iter=150,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE,
        ),
    ),
]


## 9. Boucle : fit → predict → évaluation

On entraîne chaque candidat sur le **train** et on le compare sur la **validation**.

Le test 2014 reste intact jusqu'au choix final.


In [ ]:
results = []
fitted_models = {}

for name, estimator in candidates:
    print(f"Training: {name}")

    model = clone(estimator)
    model.fit(X_train_fit, y_train_fit)       # TRAINING / FIT

    metrics = evaluate(
        name,
        model,
        X_valid_eval,
        y_valid_eval,
    )                                        # PREDICT + EVALUATION

    results.append(metrics)
    fitted_models[name] = model

    print(
        f"  MAE={metrics['MAE']:.4f} | "
        f"RMSE={metrics['RMSE']:.4f} | "
        f"R²={metrics['R2']:.4f}"
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

results_df


## 10. Choix du meilleur candidat

Ici on choisit le modèle ayant la plus petite RMSE sur **2013 (validation)**.


In [ ]:
best_name = results_df.loc[0, "model"]
best_model = fitted_models[best_name]

print("Meilleur candidat :", best_name)


## 11. Évaluation finale sur 2014

Le jeu de test n'a pas servi au choix des hyperparamètres.

C'est lui qui donne l'estimation finale de la capacité de généralisation.


In [ ]:
final_metrics = evaluate(
    best_name,
    best_model,
    X_test_eval,
    y_test_eval,
)

pd.DataFrame([final_metrics])


## 12. Exemple d'inférence

Une fois entraîné, le modèle reçoit uniquement des **features X** et prédit une **cible ŷ**.


In [ ]:
sample = X_test_eval[[0]]   # garde une forme 2D : (1, n_features)

prediction = best_model.predict(sample)

print("Prédiction :", prediction[0])
print("Valeur réelle :", y_test_eval[0])


## À retenir

```text
DATASET
   ↓
NETTOYAGE / FEATURES
   ↓
SPLIT TEMPOREL
   ├── TRAIN 2012       → .fit()
   ├── VALIDATION 2013  → choix modèle / hyperparamètres
   └── TEST 2014        → évaluation finale
                          ↓
                    MODÈLE RETENU
                          ↓
                       .predict()
```

Pour une **time series**, le split chronologique est essentiel pour éviter de laisser le modèle apprendre avec des informations du futur.
